# 🚦 Canadian Traffic Sign Classifier — Training Notebook

**Before running:**
1. Go to `Runtime → Change runtime type → T4 GPU` → Save
2. Then run all cells top to bottom (`Runtime → Run all`)
3. After training, download `traffic_sign_model.h5` from the Files panel on the left


In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✅ GPU detected: {gpus[0].name}')
else:
    print('⚠️  No GPU found — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
!pip install scikit-learn opencv-python-headless kaggle --quiet
print('✅ Dependencies installed')

In [ ]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────────
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight

print('✅ Imports done')

In [ ]:
# ── Cell 4: Load GTSRB from Kaggle ───────────────────────────────────────────
# GTSRB was removed from tensorflow_datasets so we load it directly from Kaggle.
# No Kaggle account needed — we use the public URL directly.
import os, zipfile, cv2
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

IMG_SIZE    = (32, 32)
NUM_CLASSES = 43

# ── Download directly from source (no Kaggle account needed) ─────────────────
print('⬇️  Downloading GTSRB dataset...')
!wget -q --show-progress https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Training_Images.zip -O train.zip
!wget -q --show-progress https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_Images.zip -O test.zip
!wget -q --show-progress https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_GT.zip -O test_gt.zip
print('✅ Downloads complete')

# ── Extract ───────────────────────────────────────────────────────────────────
print('📦 Extracting...')
for z in ['train.zip', 'test.zip', 'test_gt.zip']:
    with zipfile.ZipFile(z, 'r') as zf:
        zf.extractall('.')
print('✅ Extraction complete')

# ── Load training images ──────────────────────────────────────────────────────
print('🔄 Loading training images...')
train_root = 'GTSRB/Final_Training/Images'
images, labels = [], []

for class_folder in sorted(os.listdir(train_root)):
    try:
        class_id = int(class_folder)
    except ValueError:
        continue
    class_path = os.path.join(train_root, class_folder)
    for fname in os.listdir(class_path):
        if not fname.endswith('.ppm'):
            continue
        img = cv2.imread(os.path.join(class_path, fname))
        if img is None:
            continue
        img = cv2.resize(img, IMG_SIZE)
        images.append(img)
        labels.append(class_id)

# ── Load test images using ground truth CSV ───────────────────────────────────
print('🔄 Loading test images...')
import csv
test_root = 'GTSRB/Final_Test/Images'
gt_path   = 'GT-final_test.csv'

with open(gt_path, newline='') as f:
    reader = csv.DictReader(f, delimiter=';')
    for row in reader:
        img = cv2.imread(os.path.join(test_root, row['Filename']))
        if img is None:
            continue
        img = cv2.resize(img, IMG_SIZE)
        images.append(img)
        labels.append(int(row['ClassId']))

# ── Build arrays ──────────────────────────────────────────────────────────────
X = np.array(images, dtype='float32') / 255.0
y = to_categorical(np.array(labels), num_classes=NUM_CLASSES)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=np.array(labels)
)

print(f'✅ Train: {len(X_train)} | Test: {len(X_test)} | Classes: {NUM_CLASSES}')

# ── Free up disk space ────────────────────────────────────────────────────────
!rm train.zip test.zip test_gt.zip
print('🧹 Cleaned up zip files')

In [ ]:
# ── Cell 5: Build model ───────────────────────────────────────────────────────
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model

def create_cnn_model(input_shape=(32, 32, 3), num_classes=43):
    inputs = Input(shape=input_shape)

    # Block 1
    x = tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)
    x = tf.keras.layers.Dropout(0.25)(x)

    # Block 2
    x = tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)
    x = tf.keras.layers.Dropout(0.25)(x)

    # Classifier
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = create_cnn_model()
model.summary()


In [ ]:
# ── Cell 6: Class weights + tf.data pipeline ─────────────────────────────────
from sklearn.utils.class_weight import compute_class_weight

# Class weights to handle imbalance
y_train_labels    = y_train.argmax(axis=1)
class_weights     = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_labels),
    y=y_train_labels
)
class_weight_dict = dict(enumerate(class_weights))
print(f'📊 Class weights computed for {len(class_weight_dict)} classes')

# ── Augmentation layers created ONCE outside tf.function ─────────────────────
# Keras layers with internal state (like RandomRotation) cannot be created
# inside @tf.function — they must be instantiated once at the module level.
aug_rotation    = tf.keras.layers.RandomRotation(10/360, seed=42)
aug_zoom        = tf.keras.layers.RandomZoom(0.15, seed=42)
aug_translation = tf.keras.layers.RandomTranslation(0.1, 0.1, seed=42)

def augment(image, label):
    image = aug_rotation(image, training=True)
    image = aug_zoom(image, training=True)
    image = aug_translation(image, training=True)
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

# ── tf.data pipeline ──────────────────────────────────────────────────────────
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = (
    train_ds
    .shuffle(buffer_size=len(X_train), seed=42)
    .map(augment, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .repeat()       # ← makes dataset infinite, fixes 'ran out of data'
    .prefetch(AUTOTUNE)
)

val_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))
val_ds = (
    val_ds
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

steps_per_epoch = len(X_train) // BATCH_SIZE
print(f'✅ tf.data pipeline ready — {steps_per_epoch} steps/epoch')


In [ ]:
# ── Cell 7: Train ─────────────────────────────────────────────────────────────
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        'traffic_sign_model.keras',
        save_best_only=True,
        monitor='val_accuracy',
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
]

print('🚀 Training on GPU...')
history = model.fit(
    train_ds,
    steps_per_epoch=steps_per_epoch,
    epochs=50,
    validation_data=val_ds,
    class_weight=class_weight_dict,
    callbacks=callbacks,
)

best_val_acc = max(history.history['val_accuracy'])
print(f'\n✅ Training complete')
print(f'🏆 Best validation accuracy: {best_val_acc:.4f}')
print(f'🎯 Expected: >0.95 with clean GTSRB data')


In [ ]:
# ── Cell 8: Plot training curves ──────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'],     label='Train Accuracy')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True)

ax2.plot(history.history['loss'],     label='Train Loss')
ax2.plot(history.history['val_loss'], label='Val Loss')
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print('📈 Saved training_curves.png')

In [ ]:
# ── Cell 9: Download model ────────────────────────────────────────────────────
from google.colab import files

print('⬇️  Downloading model to your computer...')
files.download('traffic_sign_model.keras')
files.download('training_curves.png')
print('✅ Done — place traffic_sign_model.keras in your project folder')